# Skin-Lesion Classification on HAM10000: CNN Baseline, Transfer Learning and a Critical Comparison with DiffMIC

**CM4126 Computer Vision, Coursework 2026/27** | Dataset: [HAM10000](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000) | Reviewed paper: *[DiffMIC: Dual-Guidance Diffusion Network for Medical Image Classification](https://link.springer.com/chapter/10.1007/978-3-031-43987-2_10)* (MICCAI 2023)

## Contents

1. [Introduction](#1-introduction)
2. [Research and Data Exploration](#2-research-and-data-exploration): paper review, dataset choice, exploration, splits, image quality, enhancement
3. [Baseline Model](#3-baseline-model): training framework and a CNN trained from scratch
4. [Solution Improvement](#4-solution-improvement): transfer learning (3 pretrained models), class imbalance, image enhancement, GAN augmentation, plus a hyper-parameter search and a seed study
5. [Evaluation, Visualisation and Comparison](#5-evaluation-visualisation-and-comparison): test-set results, Grad-CAM, comparison with the baseline and with the paper
6. [Conclusion and Future Directions](#6-conclusion-and-future-directions)
7. [References](#references) and [Appendix](#appendix)

## 1. Introduction

**Problem.** Skin cancer is one of the most common cancers, and melanoma is its most dangerous form: caught early it is usually curable, caught late it is often fatal. A classifier for dermatoscopic images could act as a second reader and help prioritise patients.

**Task.** Classify a dermascopic image into one of 7 diagnostic categories (for example naevus, melanoma or basal cell carcinoma) on a highly imbalanced dataset, HAM10000.

**Objectives:**

| # | Objective | Where |
|---|---|---|
| 1 | Select and critically review a high-quality paper | §2.1 |
| 2 | Justify and describe the dataset (size, splits, classes, image quality, standardisation, enhancement) | §2.2–§2.10 |
| 3 | Build a CNN **baseline** that follows good practice, with every setting justified | §3 |
| 4 | **Improve** it: transfer learning (three pretrained models), imbalance handling, image enhancement, GAN images, a hyper-parameter search and a seed study | §4 |
| 5 | Evaluate with suitable metrics, **visualise** the results (learning curves, confusion matrices, Grad-CAM) and **compare** them with the baseline and the paper | §5 |
| 6 | Keep everything **reproducible** | whole notebook |

**Principles I followed throughout:**

1. **No data leakage.** HAM10000 often has several photos of one lesion, so I split the data by lesion, not by image. The paper's random image split is reproduced once to measure what it changes (§4.8, §5.6).
2. **Decisions use the validation set only.** The imbalance strategy, enhancement, hyper-parameters, early stopping and checkpoints never look at the test set, which is used only in §3.3 and §5.
3. **Accuracy is not enough.** 67 % of the images are one class, so always answering "naevus" is already 67 % accurate. The main metrics are **macro-F1** (the same family as the paper's F1), **balanced accuracy**, **melanoma recall** and one-vs-rest **AUC**. Per class I report recall, precision and F1.

### How to run this notebook

Open the notebook in VS Code, follow the set-up steps in the first code cell (§1.1) and press **Run All**; there is a single configuration.

* **Data:** if HAM10000 is not in `data/` yet, the first cells download it from Kaggle.
* **Saved results:** resized images and trained models are kept in `outputs/` and reloaded only if their settings and data are unchanged. Deleting `outputs/` forces a full re-run, which takes about an hour on the reference machine.
* **Reference machine:** AMD Ryzen 7 7800X3D, 64 GB RAM, AMD Radeon RX 7900 XT (20 GB), Windows 11, Python 3.12.10, PyTorch 2.9.1 (ROCm 7.2.1); speed benchmark in Appendix B.
* **Reproducibility:** every random generator uses seed 42 and the GPU kernels are deterministic. Another machine or driver can still give slightly different numbers, within the run-to-run noise measured in §4.6.

### 1.1 Set-up and configuration

All settings live in one cell, and each value is explained where it is used.

A few engineering choices make the notebook faster without changing what the models learn: a workaround for a BatchNorm bug in the Windows ROCm build of PyTorch (Appendix A); decoding the resized images once, caching them on disk and keeping them on the GPU, with augmentation on the GPU as well, because the CPU data loader was the bottleneck (Appendix B); 16-bit mixed precision, which was roughly two to five times faster than 32-bit in the benchmark; and saving trained models so that they are reused when their settings have not changed.

In [ ]:
#Set-up: create a virtual environment with Python 3.12 (e.g. `python -m venv venv`) and select it as the notebook kernel.
#Then, BEFORE running this cell, install PyTorch and torchvision built for your hardware:
#  NVIDIA, Linux:   `pip install torch torchvision` (the default build already supports CUDA).
#  NVIDIA, Windows: the default build is CPU-only; use the CUDA install command from https://pytorch.org/get-started/locally/
#  AMD, Linux:      use the ROCm install command from https://pytorch.org/get-started/locally/
#  AMD, Windows:    use AMD's ROCm build of PyTorch (see AMD's ROCm documentation); this is the set-up used here (Appendix A).
#  macOS:           `pip install torch torchvision`; current releases use the GPU only on Apple-silicon Macs (MPS), not NVIDIA or AMD.
#ROCm builds expose the GPU through torch.cuda, so NVIDIA and AMD GPUs are found the same way. Without a GPU the notebook still runs, but training takes far longer.
%pip install -q numpy pandas matplotlib opencv-python pillow scikit-learn tqdm kagglehub

In [ ]:
import os, platform, copy, hashlib, json, math, random, sys, time, warnings
from functools import partial
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# Windows + AMD only: skip a GPU-detection probe that fails when the folder path contains spaces.
# Has to happen before torch is imported (Appendix A).
if os.name == "nt" and "ROCM_SDK_TARGET_FAMILY" not in os.environ:
    try:
        from rocm_sdk import _dist_info
        if "custom" in _dist_info.AVAILABLE_TARGET_FAMILIES:
            os.environ["ROCM_SDK_TARGET_FAMILY"] = "custom"
    except Exception:
        pass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import models as tvm

from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, precision_recall_fscore_support)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from tqdm import tqdm as _tqdm
from IPython.display import display

tqdm = partial(_tqdm, file=sys.stdout, ncols=100, mininterval=1)      # plain-text progress bars, saved with the notebook

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 30, "display.width", 140)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.autolayout": True})
print(f"Python {platform.python_version()} | torch {torch.__version__} | torchvision {torchvision.__version__} | "
      f"numpy {np.__version__} | pandas {pd.__version__} | opencv {cv2.__version__}")

In [ ]:
# Settings shared by all experiments
SEED = 42

# images
IMG_SIZE = 224                           # input size of the ImageNet models
IMAGENET_MEAN = (0.485, 0.456, 0.406)    # ImageNet statistics, expected by the pretrained weights
IMAGENET_STD = (0.229, 0.224, 0.225)
N_CLASSES = 7

# training
BATCH_SIZE = 32                          # throughput stops rising here on this GPU (Appendix B)
EPOCHS_CNN = 20                          # baseline; early stopping usually ends it sooner
EPOCHS_HEAD = 3                          # transfer learning stage 1: new output layer only
EPOCHS_FINETUNE = 25                     # transfer learning stage 2: whole network
PATIENCE = 5                             # early stopping, pretrained models
PATIENCE_CNN = 5                         # the baseline learns more slowly
N_SEARCH_TRIALS = 3
SEARCH_FT_EPOCHS = 3                     # kept shot on purpose
GAN_STEPS = 5000                        # about 134 passes over the training images at batch 64; not tuned
N_BOOTSTRAP = 200                       # resamples for the 95 % confidence intervals

# reported by the paper on HAM10000 (Table 1b)
PAPER = {"name": "DiffMIC (Yang et al., MICCAI 2023)", "accuracy": 0.906, "f1": 0.816}

In [ ]:
# Download HAM10000 from Kaggle into data/ unless it is already there
import shutil, kagglehub
DATA_DIR = Path("data")
if not (DATA_DIR / "HAM10000_metadata.csv").exists():
    kaggle_dir = Path(kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000"))
    DATA_DIR.mkdir(exist_ok=True)
    for name in ["HAM10000_metadata.csv", "hmnist_8_8_L.csv", "hmnist_8_8_RGB.csv", "hmnist_28_28_L.csv", "hmnist_28_28_RGB.csv"]:
        shutil.copy2(kaggle_dir / name, DATA_DIR / name)
    for name in ["HAM10000_images_part_1", "HAM10000_images_part_2"]:
        shutil.copytree(kaggle_dir / name, DATA_DIR / name, dirs_exist_ok=True)
print("data jpg files:", len(list(DATA_DIR.rglob("*.jpg"))))

In [ ]:
DATA_DIR = Path("data")
OUT_DIR = Path("outputs/checkpoints")    # trained weights and training histories
CACHE_DIR = Path("outputs/cache")        # resized images (.npy)
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

META_CSV = DATA_DIR / "HAM10000_metadata.csv"
IMAGE_PATHS = {p.stem: p for p in DATA_DIR.rglob("*.jpg")}      # image_id -> file
print("metadata  :", META_CSV.resolve())
print("jpg files :", len(IMAGE_PATHS))

In [ ]:
def set_seed(seed):
    """Seed every random generator and make the GPU kernels deterministic."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


if torch.cuda.is_available():                      # AMD ROCm also shows up as "cuda"
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# Mixed precision (Appendix B). float16 needs a GradScaler; bfloat16 does not and is safer for transformers, so ViT uses it.
AMP_FP16 = torch.float16 if device.type == "cuda" else None
AMP_BF16 = torch.bfloat16 if device.type == "cuda" else None

set_seed(SEED)
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

In [ ]:
# Windows + AMD only (Appendix A): the ROCm build sends BatchNorm to MIOpen, whose run-time kernel compile fails with
# "'type_traits' file not found" (https://github.com/ROCm/TheRock/issues/9181). PyTorch's own GPU kernel does the same maths.
def apply_rocm_windows_batchnorm_fix():
    if os.name != "nt" or getattr(torch.version, "hip", None) is None:
        return False
    original = F.batch_norm

    def native_batch_norm(input, running_mean, running_var, weight=None, bias=None, training=False, momentum=0.1, eps=1e-5):
        if not input.is_cuda:
            return original(input, running_mean, running_var, weight, bias, training, momentum, eps)
        if training and input.numel() // max(input.shape[1], 1) <= 1:
            raise ValueError(f"Expected more than 1 value per channel when training, got {tuple(input.shape)}")
        previous = torch.backends.cudnn.enabled         # PyTorch 2.9 chooses the backend from this flag
        torch.backends.cudnn.enabled = False
        try:
            return torch.batch_norm(input, weight, bias, running_mean, running_var, training, momentum, eps, False)
        finally:
            torch.backends.cudnn.enabled = previous

    F.batch_norm = native_batch_norm
    return True


print("BatchNorm workaround for Windows + ROCm active:", apply_rocm_windows_batchnorm_fix())

In [ ]:
# Numbered captions: table captions go above the table, figure captions below the figure.
from IPython.display import Markdown

_CAPTION_COUNTER = {"Figure": 0, "Table": 0}


def caption(kind, text):
    _CAPTION_COUNTER[kind] += 1
    display(Markdown(f"**{kind} {_CAPTION_COUNTER[kind]}.** {text}"))


def show_fig(text):
    plt.show()
    caption("Figure", text)


def show_table(df, text):
    caption("Table", text)
    display(df)

## 2. Research and Data Exploration

### 2.1 Paper review

**Paper.** Yang, Y., Fu, H., Aviles-Rivero, A. I., Schönlieb, C.-B. and Zhu, L. (2023). *DiffMIC: Dual-Guidance Diffusion Network for Medical Image Classification.* MICCAI 2023, LNCS, pp. 95–105. DOI 10.1007/978-3-031-43987-2_10.

**Why this paper, and how I judged its quality.** MICCAI is a leading peer-reviewed medical-imaging conference. HAM10000 is one of the paper's three benchmarks, so its numbers are comparable with mine, and the code is public. The baselines are strong: ResNet, ViT, Swin, PVT and long-tail methods such as LDAM and ProCo. I judged quality by novelty, baseline strength, ablations and implementation detail, then read the paper critically for flaws in the evaluation.

**Method.** DiffMIC uses a diffusion model (DDPM) as a classifier: it adds noise to the label vector and learns to remove it, conditioned on the image, with a ResNet18 image embedding. Dual-granularity Conditional Guidance (DCG) adds a *global* prior from a saliency map of the whole image and a *local* prior from six 32×32 salient patches combined by gated attention. Both priors shift the noisy label and condition the denoiser, and a Maximum-Mean-Discrepancy loss (weight 0.5) pulls the predicted noise for each prior towards Gaussian noise. At inference the label is denoised from pure noise in T = 250 steps on HAM10000.

**Results.** On a 70/30 split of HAM10000, DiffMIC reports accuracy 0.906 and F1 0.816, against 0.887 and 0.763 for the best baseline, ProCo. The module ablation uses only the authors' placenta dataset, where diffusion, DCG and MMD each add roughly 1–3 accuracy points.

**Critical evaluation.**
1. *Possible leakage.* HAM10000 has 10,015 images of far fewer lesions, and the paper splits 7:3 without saying whether images of one lesion stay together, so its scores may be inflated. This is why I split by lesion.
2. *No variance.* Single runs without confidence intervals, so the gains over ProCo (0.019 accuracy, 0.053 F1) may be noise.
3. *Narrow metrics.* Only accuracy and F1 (averaging not stated); no sensitivity, AUC or melanoma recall.
4. *Weak support for some claims.* The ablation is not repeated on HAM10000, and the noise-removal claim rests on a t-SNE plot.
5. *Cost.* 1,000 epochs on an RTX 3090, and no validation protocol is described.

**What I take from it.** I use the same ResNet18 encoder, 224 px centre crop, flip and rotation augmentation and Adam (as AdamW), and report accuracy and macro-F1 for comparison. The data are split by lesion, and the paper's random split is run once to measure the leakage. Re-implementing the diffusion pipeline was beyond my compute and time, so §5.6 compares my results with the reported numbers and discusses the gap.

### 2.2 Why HAM10000

| Reason | Evidence |
|---|---|
| **Public, expert-labelled, citable** | 10,015 dermatoscopic images in 7 diagnostic classes; more than half of the diagnoses are confirmed by histopathology (Tschandl et al., 2018). |
| **Widely used benchmark** | It is the training set of ISIC 2018 Task 3 (Codella et al., 2019) and is used by the reviewed paper and the methods it compares with (DANIL, contrastive learning, ProCo), so results can be compared with the literature. |
| **Clinically important** | Melanoma detection is one of the best-known medical-imaging tasks (Esteva et al., 2017). |
| **Interesting problems** | strong class imbalance (about 58 : 1); a domain very different from ImageNet, which makes transfer learning a real question; artefacts such as hair, rulers and vignettes that motivate enhancement; and several images per lesion, which makes a leakage-free split necessary. |
| **Right size for the coursework** | about 2.8 GB and 10,000 images: every experiment trains in minutes on one GPU, and the data are still large enough for stable results. |

### 2.3 Dataset description

The metadata file has one row per image. The next cells load it and summarise the images.

In [ ]:
CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]            # alphabetical; this is the label order
CLASS_IDX = {c: i for i, c in enumerate(CLASSES)}
MEL = CLASS_IDX["mel"]
CLASS_INFO = {                                                          # code: (full name, category)
    "akiec": ("Actinic keratoses / intra-epithelial carcinoma", "pre-malignant"),
    "bcc":   ("Basal cell carcinoma", "malignant"),
    "bkl":   ("Benign keratosis-like lesions", "benign"),
    "df":    ("Dermatofibroma", "benign"),
    "mel":   ("Melanoma", "malignant"),
    "nv":    ("Melanocytic naevi", "benign"),
    "vasc":  ("Vascular lesions", "benign"),
}

meta = pd.read_csv(META_CSV)
meta["label"] = meta["dx"].map(CLASS_IDX)
meta["path"] = meta["image_id"].map(IMAGE_PATHS)
assert meta["path"].notna().all(), "some image files from the metadata were not found"

print(f"{len(meta)} images in the metadata")
show_table(meta.drop(columns=["path"]).head(), "First five rows of the metadata (one row per image).")

In [ ]:
# image sizes from the JPEG headers only, without decoding
def image_size(path):
    with Image.open(path) as im:
        return im.size                                   # (width, height)

with ThreadPoolExecutor(4) as pool:
    sizes = list(pool.map(image_size, meta["path"]))
size_counts = pd.Series(sizes).value_counts()

overview = pd.DataFrame({"value": {
    "images": len(meta),
    "distinct lesions (lesion_id)": meta["lesion_id"].nunique(),
    "images per lesion (mean)": round(len(meta) / meta["lesion_id"].nunique(), 2),
    "classes": meta["dx"].nunique(),
    "image size(s) in pixels (w x h)": ", ".join(f"{w}x{h} ({n})" for (w, h), n in size_counts.items()),
    "colour channels": "3 (RGB JPEG)",
    "images with missing age": int(meta["age"].isna().sum()),
    "total size on disk (GB)": round(sum(p.stat().st_size for p in meta["path"]) / 1e9, 2),
}})
show_table(overview, "Dataset overview: images, lesions, image size, channels, missing metadata and size on disk.")

class_table = (meta.groupby("dx").agg(images=("image_id", "count"), lesions=("lesion_id", "nunique"))
               .reindex(CLASSES))
class_table.insert(0, "category", [CLASS_INFO[c][1] for c in CLASSES])
class_table.insert(0, "diagnosis", [CLASS_INFO[c][0] for c in CLASSES])
class_table["share %"] = (100 * class_table["images"] / class_table["images"].sum()).round(1)
class_table.index.name = "class"
show_table(class_table.style.format({"share %": "{:.1f}"}).set_properties(subset=["diagnosis", "category"], **{"text-align": "left"}),
           "Images and lesions per diagnosis (the class code is the row label), with the clinical category and the share of the dataset.")

**Findings.** All images have the same format, 600 × 450 RGB JPEG. The 10,015 images show 7,470 distinct lesions, so some lesions appear more than once (Figure 1, right).

### 2.4 Class distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

counts = class_table["images"]
bars = axes[0].bar(CLASSES, counts, color=["#d95f02" if CLASS_INFO[c][1] != "benign" else "#1b9e77" for c in CLASSES])
for b, c in zip(bars, counts):
    axes[0].text(b.get_x() + b.get_width() / 2, c, f"{c}\n{100 * c / counts.sum():.1f}%", ha="center", va="bottom", fontsize=8)
axes[0].set_title("Images per class (orange = malignant / pre-malignant)")
axes[0].set_ylabel("number of images"); axes[0].set_ylim(0, counts.max() * 1.18)

axes[1].bar(CLASSES, counts, color="#7570b3"); axes[1].set_yscale("log")
axes[1].set_title("Same data, logarithmic axis"); axes[1].set_ylabel("number of images (log)")

per_lesion = meta.groupby("lesion_id").size()
axes[2].hist(per_lesion, bins=np.arange(1, per_lesion.max() + 2) - 0.5, color="#66a61e")
axes[2].set_yscale("log"); axes[2].set_xlabel("images of the same lesion"); axes[2].set_ylabel("number of lesions (log)")
axes[2].set_title("Several images of one lesion are common")
for ax in axes[:2]:
    ax.set_xlabel("diagnosis (class code, see the class table above)")
show_fig("Class distribution: images per class (left; orange = malignant or pre-malignant), the same on a log axis (middle) and how many images each lesion has (right).")

ratio = counts.max() / counts.min()
multi = (per_lesion > 1).sum()
print(f"imbalance ratio (largest / smallest class): {ratio:.1f} : 1")
print(f"majority-class accuracy (always predict 'nv'): {100 * counts.max() / counts.sum():.1f} %")
print(f"lesions photographed more than once: {multi} of {len(per_lesion)}  "
      f"({100 * meta['lesion_id'].map(per_lesion).gt(1).mean():.1f} % of all images belong to such lesions)")

**Findings.** `nv` is two thirds of the data, while `df` and `vasc` are only 1.1 % and 1.4 % (58 : 1). So accuracy alone is misleading (§1), the splits are stratified, and §4.2 compares a weighted loss with balanced sampling. Melanoma is the second-largest class (about 11 %), and I report it separately because a missed melanoma is the costliest error. 1,956 of the 7,470 lesions have more than one image, which is why the data are split by lesion (§2.6).

### 2.5 Patient and lesion metadata

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))

axes[0, 0].hist(meta["age"].dropna(), bins=range(0, 95, 5), color="#4c72b0")
axes[0, 0].set_title(f"Age (years); {int(meta['age'].isna().sum())} missing"); axes[0, 0].set_xlabel("age")

sex = meta["sex"].fillna("unknown").value_counts()
axes[0, 1].bar(sex.index, sex.values, color="#55a868"); axes[0, 1].set_title("Sex")

loc = meta["localization"].value_counts()
axes[0, 2].barh(loc.index[::-1], loc.values[::-1], color="#c44e52"); axes[0, 2].set_title("Body location")

dxt = meta["dx_type"].value_counts()
axes[1, 0].bar(dxt.index, dxt.values, color="#8172b2"); axes[1, 0].set_title("How the diagnosis was confirmed")
axes[1, 0].tick_params(axis="x", rotation=20)

data = [meta.loc[meta["dx"] == c, "age"].dropna() for c in CLASSES]
axes[1, 1].boxplot(data, tick_labels=CLASSES, showfliers=False); axes[1, 1].set_title("Age by class")

tab = pd.crosstab(meta["dx"], meta["dx_type"]).reindex(CLASSES)
(tab.div(tab.sum(axis=1), axis=0) * 100).plot(kind="bar", stacked=True, ax=axes[1, 2], legend=True, width=0.8)
axes[1, 2].set_title("Confirmation type within each class (%)"); axes[1, 2].set_xlabel(""); axes[1, 2].tick_params(axis="x", rotation=0)
axes[1, 2].legend(fontsize=7)
show_fig("Metadata overview: age, sex, body location, how the diagnosis was confirmed, age per class and confirmation type per class.")

**Findings.** Like the paper, I treat this as pure image classification and do not feed the metadata to the models. Age differs by class (melanoma, carcinomas and keratoses are more common in older patients), which the models do not control for. Most lesions are on the back, lower extremity and trunk. Some diagnoses rest on follow-up or consensus rather than histopathology, so part of the labels are expert opinion. The 57 missing ages do not matter because age is not used.

### 2.6 Train / validation / test splits

**Every lesion goes to exactly one split.** `StratifiedGroupKFold` with `groups = lesion_id` and `y = diagnosis` keeps all images of a lesion together and the class proportions about equal; two successive splits give about **70 % train, 15 % validation and 15 % test**. Validation drives every decision and the test set is only used for reporting (§3.3, §5). For the leakage experiment (§4.8) I also build the paper's protocol, a random 70/30 split of *images*, holding out 10 % of its training part for validation.

In [ ]:
def lesion_level_split(df, seed):
    """70/15/15 split, stratified by diagnosis and grouped by lesion. Returns row positions."""
    outer = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=seed)          # 1/7 of the data -> test
    trainval, test = next(outer.split(df, df["dx"], groups=df["lesion_id"]))
    tv = df.iloc[trainval]
    inner = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=seed)          # 1/6 of the rest -> validation
    tr, va = next(inner.split(tv, tv["dx"], groups=tv["lesion_id"]))
    return trainval[tr], trainval[va], test


def paper_style_split(df, seed):
    """The paper's protocol: random 70/30 split of images (lesions ignored); 10 % of train held out for validation."""
    idx = np.arange(len(df))
    train, test = train_test_split(idx, test_size=0.30, stratify=df["dx"], random_state=seed)
    train, val = train_test_split(train, test_size=0.10, stratify=df["dx"].iloc[train], random_state=seed)
    return train, val, test


SPLITS = {}
for name, fn in [("lesion", lesion_level_split), ("paper", paper_style_split)]:
    tr, va, te = fn(meta, SEED)
    SPLITS[name] = {"train": np.sort(tr), "val": np.sort(va), "test": np.sort(te)}

# no lesion may appear in two parts of the lesion-level split
L = SPLITS["lesion"]
lesions = {k: set(meta["lesion_id"].iloc[v]) for k, v in L.items()}
assert not (lesions["train"] & lesions["val"]) and not (lesions["train"] & lesions["test"]) and not (lesions["val"] & lesions["test"])
print("lesion-level split: no lesion is shared between train, validation and test.\n")

# number of paper-style test images that show a lesion also in its training part
P = SPLITS["paper"]
train_lesions = set(meta["lesion_id"].iloc[P["train"]])
leaked = meta["lesion_id"].iloc[P["test"]].isin(train_lesions)
print(f"paper-style split: {leaked.sum()} of {len(P['test'])} test images ({100 * leaked.mean():.1f} %) show a lesion "
      f"that is also in the training set.\n")

rows = []
for proto in SPLITS:
    for part, idx in SPLITS[proto].items():
        c = meta["dx"].iloc[idx].value_counts().reindex(CLASSES).fillna(0).astype(int)
        rows.append({"protocol": proto, "split": part, "images": len(idx), "lesions": meta["lesion_id"].iloc[idx].nunique(),
                     **{k: v for k, v in c.items()}})
split_table = pd.DataFrame(rows)
show_table(split_table, "Images, lesions and class counts in the train, validation and test part of both split protocols.")

In [ ]:
# class proportions per split for both protocols, and the share of leaked test images
fig, axes = plt.subplots(1, 3, figsize=(16, 3.9), gridspec_kw={"width_ratios": [2.3, 2.3, 1.1]})
for ax, proto in zip(axes[:2], ["lesion", "paper"]):
    props = pd.DataFrame({part: meta["dx"].iloc[idx].value_counts(normalize=True).reindex(CLASSES).fillna(0) * 100
                          for part, idx in SPLITS[proto].items()})
    props.plot(kind="bar", ax=ax, width=0.8)
    ax.set_title(f"Class proportions per split ({proto} protocol)"); ax.set_ylabel("% of the split")
    ax.set_xlabel("diagnosis (class code)"); ax.tick_params(axis="x", rotation=0)

leak_share = {}
for proto in SPLITS:                                        # test images whose lesion is also in the training part
    train_l = set(meta["lesion_id"].iloc[SPLITS[proto]["train"]])
    leak_share[proto] = 100 * meta["lesion_id"].iloc[SPLITS[proto]["test"]].isin(train_l).mean()
axes[2].bar(["lesion\nprotocol", "paper\nprotocol"], [leak_share["lesion"], leak_share["paper"]], color=["#1b9e77", "#d95f02"])
for k, proto in enumerate(("lesion", "paper")):
    axes[2].text(k, leak_share[proto] + 1, f"{leak_share[proto]:.1f} %", ha="center")
axes[2].set_ylim(0, 50); axes[2].set_ylabel("% of test images with a lesion\nthat is also in the training set"); axes[2].set_title("Leakage into the test set")
show_fig("Left and middle: class proportions in the train, validation and test part of both split protocols (both are stratified, so the bars are almost equal by design). "
         f"Right: share of the test images whose lesion also appears in the training part ({leak_share['lesion']:.0f} % for the lesion-level protocol, {leak_share['paper']:.0f} % for the paper-style protocol).")

**Findings.** Both protocols are stratified, so the class mix is the same in every part and the left and middle panels look alike by design. The right panel shows the real difference: no test image of the lesion-level split shows a lesion from the training set, against 33 % (992 of 3,005) for the paper-style split. These near-duplicates are the leakage studied in §4.8 and §5.6. Everything below uses the lesion-level split unless stated otherwise.

### 2.7 Loading, centre-cropping and caching the images

Pretrained networks expect 224 × 224 input, and the images are 4:3. Like the paper, I **centre-crop the largest square (450 × 450) and resize it to 224 × 224** with Lanczos filtering. This keeps the lesion's proportions, and lesions sit near the centre anyway. Decoding 10,015 JPEGs in every epoch would take longer than the training itself, so the images are decoded **once**, saved as a `uint8` array (1.5 GB) and kept on the GPU (Appendix B).

In [ ]:
def load_square_224(path, size=IMG_SIZE):
    """Centre-crop the largest square of a JPEG and resize it; returns an RGB uint8 array."""
    with Image.open(path) as im:
        im = im.convert("RGB")
        w, h = im.size
        s = min(w, h)
        left, top = (w - s) // 2, (h - s) // 2
        return np.asarray(im.crop((left, top, left + s, top + s)).resize((size, size), Image.BILINEAR), dtype=np.uint8)


def cached_array(tag, ids, build_fn):
    """Load the uint8 array [N, H, W, 3] from CACHE_DIR, or build it with build_fn and save it."""
    key = hashlib.md5("|".join(ids).encode()).hexdigest()[:8]
    file = CACHE_DIR / f"cache_{tag}_{IMG_SIZE}_{len(ids)}_{key}.npy"
    if file.exists():
        print(f"[{tag}] loaded from {file.name}")
        return np.load(file)
    t0 = time.perf_counter()
    arr = build_fn()
    np.save(file, arr)
    print(f"[{tag}] built {arr.shape} in {time.perf_counter() - t0:.0f} s and saved to {file.name}")
    return arr


def _build_raw():
    with ThreadPoolExecutor(4) as pool:                                    # PIL releases the GIL while decoding
        return np.stack(list(tqdm(pool.map(load_square_224, meta["path"]), total=len(meta), desc="decoding images")))


IDS = meta["image_id"].tolist()
X_RAW = cached_array("raw", IDS, _build_raw)
Y = meta["label"].to_numpy()
print("raw cache:", X_RAW.shape, X_RAW.dtype, f"{X_RAW.nbytes / 1e9:.2f} GB")

### 2.8 Looking at the data and assessing image quality

Following the Week 6 advice to look at the data before training: five random images per class after cropping and resizing.

In [ ]:
def show_gallery(x, labels, per_class=5, seed=SEED, title=""):
    rng = np.random.RandomState(seed)
    fig, axes = plt.subplots(len(CLASSES), per_class, figsize=(2.0 * per_class, 2.05 * len(CLASSES)))
    for r, c in enumerate(CLASSES):
        pick = rng.choice(np.flatnonzero(labels == CLASS_IDX[c]), size=per_class, replace=False)
        for k in range(per_class):
            ax = axes[r, k]; ax.axis("off")
            if k < len(pick):
                ax.imshow(x[pick[k]])
            if k == 0:
                ax.set_title(f"{c}: {CLASS_INFO[c][0][:30]}", fontsize=8, loc="left")
    fig.suptitle(title, y=1.0); show_fig(title or "Random images of every class")


show_gallery(X_RAW, Y, title="Five random images per class (224 x 224, centre-cropped)")

**Observations.** Many images contain **hair** over the lesion; some have a dark circular vignette or are slightly blurred, and colour and lighting vary between devices and clinics. `mel`, `nv` and `bkl` often look alike.

Five **quality measures** per image (224 px version) put numbers on this: brightness (mean grey level), contrast (grey-level standard deviation), sharpness (variance of the Laplacian), the share of **dark corner** pixels (vignette) and the share of **hair** pixels (black-hat filter plus an elongation test).

In [ ]:
_HAIR_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))          # black-hat kernel for thin dark structures


def hair_mask(img_rgb, threshold=20, min_length=30, max_thickness=4.0):
    """Hair = thin dark structures (black-hat filter) that are also long and thin.

    A plain black-hat threshold also fires on pigment-network lines and dots inside the lesion (about 3 % of all
    pixels, see the check after the hair-removal figure), so a connected component is kept only if it is at least
    min_length pixels long and at most max_thickness pixels thick on average (area / length).
    """
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    dark = (cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, _HAIR_KERNEL) > threshold).astype(np.uint8)
    _, comp, stats, _ = cv2.connectedComponentsWithStats(dark, connectivity=8)
    length = np.maximum(stats[:, cv2.CC_STAT_WIDTH], stats[:, cv2.CC_STAT_HEIGHT])
    keep = (length >= min_length) & (stats[:, cv2.CC_STAT_AREA] / np.maximum(length, 1) <= max_thickness)
    keep[0] = False                                                     # label 0 is the background
    return keep[comp].astype(np.uint8)


def quality_features(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    k = 40                                                                  # 40 x 40 corner squares
    corners = np.concatenate([gray[:k, :k].ravel(), gray[:k, -k:].ravel(), gray[-k:, :k].ravel(), gray[-k:, -k:].ravel()])
    return (gray.mean(), gray.std(), cv2.Laplacian(gray, cv2.CV_64F).var(), (corners < 25).mean(), hair_mask(img).mean())


quality = pd.DataFrame([quality_features(im) for im in tqdm(X_RAW, desc="quality")],
                       columns=["brightness", "contrast", "sharpness", "dark_corner_frac", "hair_frac"])
quality["dx"] = meta["dx"].values
q_summary = quality.groupby("dx").mean(numeric_only=True).reindex(CLASSES)
q_summary[["dark_corner_frac", "hair_frac"]] *= 100                                  # as % of the pixels
q_summary = q_summary.rename(columns={"dark_corner_frac": "dark corner %", "hair_frac": "hair %"})
q_summary.index.name = "class"
show_table(q_summary.round(2), "Mean image-quality measures per diagnosis (dark corners and hair as % of the pixels).")

fig, axes = plt.subplots(1, 5, figsize=(17, 3.2))
for ax, col in zip(axes, ["brightness", "contrast", "sharpness", "dark_corner_frac", "hair_frac"]):
    # sharpness spans several orders of magnitude, so it gets log-spaced bins
    bins = np.logspace(np.log10(max(quality[col].min(), 1e-3)), np.log10(quality[col].max()), 40) if col == "sharpness" else 40
    ax.hist(quality[col], bins=bins, color="#4c72b0"); ax.set_title(col)
    ax.set_xlabel(col + (" (log scale)" if col == "sharpness" else ""))
axes[2].set_xscale("log"); axes[0].set_ylabel("number of images")
show_fig("Distribution of the five image-quality measures over all images (sharpness on a log axis).")

In [ ]:
# the worst examples of each problem, then how common it is (rough thresholds)
def show_examples(order, title, n=6):
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.5))
    for ax, i in zip(axes, order[:n]):
        ax.imshow(X_RAW[i]); ax.axis("off"); ax.set_title(f"{meta['dx'][i]}", fontsize=8)
    fig.suptitle(title, y=1.02); show_fig(title)


show_examples(np.argsort(quality["sharpness"].values), "Blurriest images (lowest Laplacian variance)")
show_examples(np.argsort(-quality["hair_frac"].values), "Most hair (highest black-hat fraction)")
show_examples(np.argsort(-quality["dark_corner_frac"].values), "Strongest dark vignette")
for t in (0.005, 0.01, 0.03):
    print(f"share of images whose hair fraction exceeds {100 * t:.1f} %: {100 * (quality['hair_frac'] > t).mean():.1f} %")
print(f"share of images with a dark vignette (> 20 % dark corner pixels): {100 * (quality['dark_corner_frac'] > 0.2).mean():.1f} %")
print(f"share of images with very low sharpness (Laplacian variance < 20): {100 * (quality['sharpness'] < 20).mean():.1f} %")

**Quality assessment and decisions.**

* **Size and format** are identical for every image; none is rejected.
* **Sharpness, brightness and contrast** vary between clinics, but no image is unusable, so I **keep all images**. Removing hard cases would make the test set unrealistically easy, and the brightness and contrast jitter of the augmentation (§3.1) covers the variation.
* **Hair**: 14.1 % of the images have more than 0.5 % hair pixels. The measure is a proxy (long, thin dark structures) that can also catch thin dark lesion borders. Hair removal is tested in §2.9 and §4.3.
* **Dark vignettes** almost disappear after the centre crop (0.2 % of the images).
* **The quality measures differ between classes** (Table 5): mean contrast is 20-22 for `akiec`, `bcc` and `df` but 28-34 for `nv`, `mel` and `vasc`, and mean sharpness ranges from 109 (`vasc`) to 203 (`bcc`). A model could exploit this bias; Grad-CAM (§5.4) checks where it looks.

### 2.9 Image enhancement

Two classical techniques from Week 1, applied once and cached:

1. **Hair removal**, similar to *DullRazor* (Lee et al., 1997): a black-hat filter finds thin dark structures, an elongation test keeps the long thin ones, and the dilated mask is filled by **inpainting** (Telea, OpenCV).
2. **CLAHE** (Zuiderveld, 1994) on the LAB lightness channel: it raises *local* contrast without amplifying noise as global histogram equalisation (Lab 5) does, and leaves the colours unchanged.